In [2]:
# import os
# # from google.colab import drive

# # 1. 경로 초기화 및 폴더 재생성
# os.chdir('/content')
# !rm -rf TrackNetV3
# !git clone https://github.com/qaz812345/TrackNetV3.git
# os.chdir('/content/TrackNetV3')

# # 2. 라이브러리 설치
# !sed -i 's/numpy==1.22.4/numpy/g' requirements.txt
# !pip install -r requirements.txt
# !pip install parse gdown ultralytics opencv-python-headless

# # 3. 가중치 파일 다운로드
# !gdown 1CfzE87a0f6LhBp0kniSl1-89zaLCZ8cA
# !unzip -o TrackNetV3_ckpts.zip

# print("\n🎉 [1단계 완료] 환경 세팅 성공")

# --------------------------------------------------------------------------
# vscode에서 실행하기 위한 코드입니다. Colab에서 실행할 때는 위의 코드를 사용해주세요.
# git clone https://github.com/qaz812345/TrackNetV3.git
# pip install numpy pandas matplotlib scipy opencv-python ultralytics gdown parse ffmpeg-python


In [ ]:
# from google.colab import files
import os

# 1. 영상 업로드
print("🎯 분석할 영상을 선택해주세요.")
# uploaded = files.upload()

# original_name = list(uploaded.keys())[0]
target_video = "./inputVideo/test_8sec.mp4"
# if os.path.exists(target_video): os.remove(target_video)
# os.rename(original_name, target_video)

if not os.path.exists(target_video):
    print(f"❌ 에러: {target_video} 파일이 폴더에 없습니다! 영상을 이 폴더로 옮겨주세요.")
else:
    print(f"✅ {target_video} 연결 완료. 분석을 시작합니다.")

# 2. predict.py 패치 (num_workers=0, batch_size=1)

# with open('TrackNetV3/predict.py', 'r') as f: lines = f.readlines()
# with open('TrackNetV3/predict.py', 'w') as f:
#     for line in lines:
#         if 'num_workers=' in line:
#             prefix = line.split('num_workers=')[0]
#             suffix = line.split('num_workers=')[1].split(',')[1] if ',' in line.split('num_workers=')[1] else ')'
#             line = f"{prefix}num_workers=0, {suffix}\n"
#         if 'batch_size=' in line and '16' in line:
#             line = line.replace('16', '1')
#         f.write(line)


# 현재 내 노트북 사양에 맞춰 수동 조절
# VRAM 4GB 이하 (GTX 1650 등): 그대로 1 추천.
# VRAM 6GB ~ 8GB (RTX 3060 등): 4 또는 8로 올려보세요.
# VRAM 10GB 이상: 16 원복 가능.
NEW_BATCH_SIZE = '1' # 1보다는 빠르고 16보다는 안전한 중간값

with open('TrackNetV3/predict.py', 'r') as f: lines = f.readlines()
with open('TrackNetV3/predict.py', 'w') as f:
    for line in lines:
        if 'num_workers=' in line:
            # 일꾼은 0으로 고정 (Windows 에러 방지)
            line = line.replace('num_workers=16', 'num_workers=0').replace('num_workers=4', 'num_workers=0')
        if 'batch_size=' in line:
            # 배치 사이즈만 사양에 맞춰 조절
            import re
            line = re.sub(r'batch_size=\d+', f'batch_size={NEW_BATCH_SIZE}', line)
        f.write(line)

# 3. 분석 실행
!python TrackNetV3/predict.py \
  --video_file "./inputVideo/test_8sec.mp4" \
  --tracknet_file TrackNetV3/ckpts/TrackNet_best.pt \
  --inpaintnet_file TrackNetV3/ckpts/InpaintNet_best.pt \
  --save_dir prediction \
  --output_video

print("\n✅ [2단계 완료] TrackNetV3 좌표 데이터 추출 성공")

🎯 분석할 영상을 선택해주세요.
✅ ./inputVideo/test_8sec.mp4 연결 완료. 분석을 시작합니다.
Done.

✅ [2단계 완료] TrackNetV3 좌표 데이터 추출 성공



100%|██████████| 15/15 [05:11<00:00, 20.77s/it]

100%|██████████| 14/14 [00:00<00:00, 48.09it/s]
OpenCV: FFMPEG: tag 0x34363268/'h264' is not supported with codec id 27 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x31637661/'avc1'

Failed to load OpenH264 library: openh264-1.8.0-win64.dll
	Please check environment and/or download library: https://github.com/cisco/openh264/releases

[libopenh264 @ 0000020bb977d300] Incorrect library version loaded
[ERROR:0@319.979] global cap_ffmpeg_impl.hpp:3514 open Could not open codec libopenh264, error: Unspecified error (-22)
[ERROR:0@319.979] global cap_ffmpeg_impl.hpp:3531 open VIDEOIO/FFMPEG: Failed to initialize VideoWriter


In [ ]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from collections import deque

# 1. 영상 소스 및 데이터 로드
VIDEO_PATH = './inputVideo/test_8sec.mp4'
CSV_PATH = './prediction/test_8sec_ball.csv'
ball_df = pd.read_csv(CSV_PATH)
ball_df.columns = [c.strip().lower() for c in ball_df.columns]

cap = cv2.VideoCapture(VIDEO_PATH)
W_VAL = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_VAL = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 2. 호모그래피 설정 (미니맵)
src_pts = np.array([[W_VAL*0.28, H_VAL*0.46], [W_VAL*0.72, H_VAL*0.46], [W_VAL*0.85, H_VAL*0.97], [W_VAL*0.15, H_VAL*0.97]], dtype=np.float32)
dst_pts_base = (src_pts - [0, H_VAL*0.4]).astype(np.float32)
shifted_dst_pts = (dst_pts_base + [0, 400]).astype(np.float32)
new_matrix = cv2.getPerspectiveTransform(src_pts, shifted_dst_pts)
new_net_y = (shifted_dst_pts[0][1] + shifted_dst_pts[3][1]) / 2

# 3. 모델 로드
pose_model = YOLO('yolov8n-pose.pt')

def draw_pro_court(canvas, pts):
    RED, YELLOW = (0, 0, 255), (0, 255, 255)
    def get_line(p1, p2, ratio): return p1 + (p2 - p1) * ratio
    p1, p2, p3, p4 = pts[0], pts[1], pts[2], pts[3]
    for r in [0, 0.1, 0.38, 0.62, 0.9, 1.0]:
        l, r_p = get_line(p1, p4, r), get_line(p2, p3, r)
        cv2.line(canvas, tuple(l.astype(int)), tuple(r_p.astype(int)), RED, 2)
    for r in [0, 0.05, 0.5, 0.95, 1.0]:
        t, b = get_line(p1, p2, r), get_line(p4, p3, r)
        cv2.line(canvas, tuple(t.astype(int)), tuple(b.astype(int)), RED, 2)
    NET_H = 250
    m_l, m_r = get_line(p1, p4, 0.5), get_line(p2, p3, 0.5)
    nt_l, nt_r = (int(m_l[0]), int(m_l[1]-NET_H)), (int(m_r[0]), int(m_r[1]-NET_H))
    cv2.line(canvas, nt_l, nt_r, YELLOW, 6); cv2.line(canvas, tuple(m_l.astype(int)), nt_l, YELLOW, 3); cv2.line(canvas, tuple(m_r.astype(int)), nt_r, YELLOW, 3)

print("✅ [3단계 완료] 시각화 파라미터 준비 완료")

✅ [3단계 완료] 시각화 파라미터 준비 완료


In [26]:
import cv2
import numpy as np
import pandas as pd
from collections import deque
import os

# --- [설정 확인] ---
# 이전 셀에서 정의된 변수들이 메모리에 있어야 합니다: W_VAL, H_VAL, fps, ball_df, pose_model 등
# 만약 에러가 난다면 이전 셀들을 순서대로 다시 실행해 주세요.

# 1. 출력 설정
output_path = './temp_res.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
# 최종 영상 크기: 가로(W_VAL), 세로(H_VAL + 미니맵 1000)
out = cv2.VideoWriter(output_path, fourcc, fps, (W_VAL, H_VAL + 1000))

shuttle_history = deque(maxlen=30)
EDGES = [[16,14],[14,12],[17,15],[15,13],[12,13],[6,12],[7,13],[6,7],[6,8],[7,9],[8,10],[9,11]]

print("🚀 관절 인식 및 궤적 합성 시작 (GTX 1650 Ti 최적화)")
print(f"📊 총 처리할 프레임 수: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")

frame_idx = 0
cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # 영상 처음부터 읽기

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        break

    # [A] YOLOv8-pose 스켈레톤 추출
    # 로컬 사양에 맞춰 conf(신뢰도)를 0.3으로 설정
    pose_results = pose_model.track(frame, persist=True, verbose=False, conf=0.3)
    annotated_frame = pose_results[0].plot()
    
    # 중요: vstack을 위해 메인 프레임 크기를 원본 사이즈로 강제 고정
    annotated_frame = cv2.resize(annotated_frame, (W_VAL, H_VAL))

    # [B] TrackNetV3 데이터 기반 셔틀콕 그리기
    if frame_idx < len(ball_df):
        row = ball_df.iloc[frame_idx]
        # CSV 컬럼명이 대문자일 경우를 대비해 get 사용
        nx = row.get('x', row.get('X', 0))
        ny = row.get('y', row.get('Y', 0))

        if nx > 0 and ny > 0:
            curr_pt = (int(nx), int(ny))
            if shuttle_history:
                lx, ly = shuttle_history[-1]
                # 튐 방지: 이전 좌표와 너무 멀면 무시 (250픽셀 기준)
                if np.sqrt((nx-lx)**2 + (ny-ly)**2) < 250:
                    shuttle_history.append(curr_pt)
            else:
                shuttle_history.append(curr_pt)
            
            # 메인 영상에 노란색 점 표시
            cv2.circle(annotated_frame, curr_pt, 6, (0, 255, 255), -1)

    # [C] 미니맵 생성
    mini_map = np.zeros((1000, W_VAL, 3), dtype=np.uint8)
    draw_pro_court(mini_map, shifted_dst_pts)

    # 미니맵에 셔틀콕 궤적 투영
    pts = list(shuttle_history)
    for i in range(len(pts)):
        # 이동 평균 필터로 궤적을 부드럽게
        start_idx, end_idx = max(0, i-1), min(len(pts), i+2)
        subset = pts[start_idx:end_idx]
        avg_x = sum(p[0] for p in subset)/len(subset)
        avg_y = sum(p[1] for p in subset)/len(subset)
        
        # 호모그래피 행렬로 미니맵 좌표 변환
        p_map = cv2.perspectiveTransform(np.array([[[avg_x, avg_y]]], dtype=np.float32), new_matrix)[0][0]
        # 꼬리 효과: 시간이 흐른 좌표일수록 진하게 표시
        cv2.circle(mini_map, (int(p_map[0]), int(p_map[1])), int(3 + 5 * (i+1)/len(pts)), (255, 0, 0), -1)

    # [D] 스켈레톤 미니맵 투영
    if pose_results[0].keypoints is not None:
        try:
            kpts = pose_results[0].keypoints.data.cpu().numpy()
            for person in kpts:
                # 양 발목의 중앙점을 기준으로 위치 파악
                f_x, f_y = (person[15][0] + person[16][0]) / 2, (person[15][1] + person[16][1]) / 2
                p_dst = cv2.perspectiveTransform(np.array([[[f_x, f_y]]], dtype=np.float32), new_matrix)[0][0]
                
                # 코트(다각형) 안에 있는 사람만 그리기
                if cv2.pointPolygonTest(shifted_dst_pts.astype(np.int32), (p_dst[0], p_dst[1]), False) >= 0:
                    tx, ty = p_dst[0], p_dst[1]
                    # 네트 기준 위/아래 팀 컬러 구분
                    color = (255, 0, 255) if ty < new_net_y else (0, 255, 0)
                    
                    for edge in EDGES:
                        p1_i, p2_i = edge[0]-1, edge[1]-1
                        if person[p1_i][2] > 0.3 and person[p2_i][2] > 0.3:
                            x1, y1 = int(tx+(person[p1_i][0]-f_x)*1.2), int(ty+(person[p1_i][1]-f_y)*1.2)
                            x2, y2 = int(tx+(person[p2_i][0]-f_x)*1.2), int(ty+(person[p2_i][1]-f_y)*1.2)
                            cv2.line(mini_map, (x1, y1), (x2, y2), color, 6)
        except Exception:
            pass # 인식 오류 시 해당 프레임 건너뜀

    # [E] 최종 프레임 합치기 및 저장
    try:
        # 두 이미지의 가로 너비가 반드시 같아야 vstack이 성공합니다.
        final_frame = np.vstack((annotated_frame, mini_map))
        out.write(final_frame)
    except Exception as e:
        print(f"⚠️ 프레임 합성 실패 (프레임 {frame_idx}): {e}")
        break

    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"⏳ {frame_idx} 프레임 진행 중...")

# 자원 해제 (매우 중요)
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"✅ [4단계 완료] 총 {frame_idx} 프레임 합성 완료! (temp_res.mp4 생성됨)")

🚀 관절 인식 및 궤적 합성 시작 (GTX 1650 Ti 최적화)
📊 총 처리할 프레임 수: 239
⏳ 50 프레임 진행 중...
⏳ 100 프레임 진행 중...
⏳ 150 프레임 진행 중...
⏳ 200 프레임 진행 중...
✅ [4단계 완료] 총 239 프레임 합성 완료! (temp_res.mp4 생성됨)


In [ ]:
# out = cv2.VideoWriter('./temp_res.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (W_VAL, H_VAL + 1000))
# shuttle_history = deque(maxlen=30)
# EDGES = [[16,14],[14,12],[17,15],[15,13],[12,13],[6,12],[7,13],[6,7],[6,8],[7,9],[8,10],[9,11]]

# print("🚀 관절 인식 및 궤적 합성 중...")
# frame_idx = 0
# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret: break

#     # [A] YOLOv8-pose 스켈레톤 추출
#     pose_results = pose_model.track(frame, persist=True, verbose=False, conf=0.3)
#     annotated_frame = pose_results[0].plot()

#     # [B] TrackNetV3 CSV 데이터 기반 셔틀콕 합성
#     if frame_idx < len(ball_df):
#         row = ball_df.iloc[frame_idx]
#         nx, ny = row.get('x', row.get('X', 0)), row.get('y', row.get('Y', 0))

#         if nx > 0 and ny > 0:
#             if shuttle_history:
#                 lx, ly = shuttle_history[-1]
#                 if np.sqrt((nx-lx)**2 + (ny-ly)**2) < 250: shuttle_history.append((int(nx), int(ny)))
#             else: shuttle_history.append((int(nx), int(ny)))
#             cv2.circle(annotated_frame, (int(nx), int(ny)), 6, (0, 255, 255), -1)

#     # [C] 미니맵 및 궤적 보정 (이동 평균 필터)
#     mini_map = np.zeros((1000, W_VAL, 3), dtype=np.uint8)
#     draw_pro_court(mini_map, shifted_dst_pts)

#     pts = list(shuttle_history)
#     for i in range(len(pts)):
#         start_idx, end_idx = max(0, i-1), min(len(pts), i+2)
#         subset = pts[start_idx:end_idx]
#         avg_x, avg_y = sum(p[0] for p in subset)/len(subset), sum(p[1] for p in subset)/len(subset)
#         p_map = cv2.perspectiveTransform(np.array([[[avg_x, avg_y]]], dtype=np.float32), new_matrix)[0][0]
#         cv2.circle(mini_map, (int(p_map[0]), int(p_map[1])), int(3 + 5 * (i+1)/len(pts)), (255, 0, 0), -1)

#     # [D] 스켈레톤 미니맵 투영
#     if pose_results[0].keypoints is not None:
#         kpts = pose_results[0].keypoints.data.cpu().numpy()
#         for person in kpts:
#             f_x, f_y = (person[15][0] + person[16][0]) / 2, (person[15][1] + person[16][1]) / 2
#             p_dst = cv2.perspectiveTransform(np.array([[[f_x, f_y]]], dtype=np.float32), new_matrix)[0][0]
#             if cv2.pointPolygonTest(shifted_dst_pts.astype(np.int32), (p_dst[0], p_dst[1]), False) >= 0:
#                 tx, ty = p_dst[0], p_dst[1]
#                 color = (255, 0, 255) if ty < new_net_y else (0, 255, 0)
#                 for edge in EDGES:
#                     p1_i, p2_i = edge[0]-1, edge[1]-1
#                     if person[p1_i][2] > 0.3 and person[p2_i][2] > 0.3:
#                         x1, y1 = int(tx+(person[p1_i][0]-f_x)*1.2), int(ty+(person[p1_i][1]-f_y)*1.2)
#                         x2, y2 = int(tx+(person[p2_i][0]-f_x)*1.2), int(ty+(person[p2_i][1]-f_y)*1.2)
#                         cv2.line(mini_map, (x1, y1), (x2, y2), color, 6)

#     out.write(np.vstack((annotated_frame, mini_map)))
#     frame_idx += 1

# cap.release()
# out.release()
# print("✅ [4단계 완료] 영상 합성 완료")

🚀 관절 인식 및 궤적 합성 중...
✅ [4단계 완료] 영상 합성 완료


In [27]:
import base64
# from IPython.display import HTML/

# 1. 결과 저장 폴더 생성 (이미 있으면 통과)
os.makedirs('./result', exist_ok=True)

# !ffmpeg -y -i ./temp_res.mp4 -vcodec libx264 -f mp4 ./result/final_rally_analysis.mp4 -loglevel quiet
if os.path.getsize('./temp_res.mp4') < 1000:
    print("❌ 에러: temp_res.mp4 파일이 너무 작거나 비어있습니다. 이전 셀을 다시 확인하세요.")
else:
    # static_ffmpeg의 run 기능을 직접 호출
    !static_ffmpeg -y -i ./temp_res.mp4 -vcodec libx264 -crf 23 -pix_fmt yuv420p ./result/final_analysis.mp4
    print("✅ 최종 결과물이 ./result/final_analysis.mp4 에 저장되었습니다.")
# files.download('/content/final_rally_analysis.mp4')

# 코랩에서 바로 보기
# mp4 = open('/content/final_rally_analysis.mp4', 'rb').read()
# data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()
# display(HTML(f'<video width=800 controls><source src="{data_url}" type="video/mp4"></video>'))

✅ 최종 결과물이 ./result/final_analysis.mp4 에 저장되었습니다.


ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb --ena